<a target="_blank" href="https://colab.research.google.com/github/mim-ml-teaching/public-rl-2025-26/blob/main/labs/RL_LAB2_behavioral_cloning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 02: Behavioral Cloning

In this lab, we look into the problem of learning from expert demonstrations.

- Find a policy $\pi(a | s)$ that best imitates the expert policy $\pi^*(a | s)$ in the given environment.
- It's worth noting, that we don't need access to the environment rewards.

Major Imitation Learning techniques are:

1. Behavioural Cloning,
1. Imitation Learning via Interactive Demonstrator e.g. SMILe (Ross and Bagnell, 2010) or DAgger (Ross et al., 2011),
1. Inverse Reinforcement Learning – out of scope of this lab.

We will solve the Ant problem, shown below, examining the first two approaches.

## Install dependencies

In [ ]:
!pip -q install gymnasium[mujoco]
!pip -q install pyglet
!pip -q install pyopengl
!pip -q install pyvirtualdisplay
!pip install -q sample-factory
!pip install -q gymnasium[mujoco] mujoco
!git clone https://github.com/alex-petrenko/sample-factory.git

%cd sample-factory

## Download Expert

In [ ]:
!python -m sample_factory.huggingface.load_from_hub -r LLParallax/sf_Ant

In [ ]:
import functools

import torch

from sample_factory.algo.learning.learner import Learner
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.rl_utils import prepare_and_normalize_obs
from sample_factory.cfg.arguments import load_from_checkpoint
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config


def create_expert(cfg):
    cfg = load_from_checkpoint(cfg)

    cfg.num_envs = 1

    env = make_env_func_batched(
        cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=None
    )

    if hasattr(env.unwrapped, "reset_on_init"):
        # reset call ruins the demo recording for VizDoom
        env.unwrapped.reset_on_init = False

    actor_critic = create_actor_critic(cfg, env.observation_space, env.action_space)
    actor_critic.eval()

    device = torch.device("cpu" if cfg.device == "cpu" else "cuda")
    actor_critic.model_to_device(device)

    policy_id = cfg.policy_index
    name_prefix = dict(latest="checkpoint", best="best")[cfg.load_checkpoint_kind]
    checkpoints = Learner.get_checkpoints(Learner.checkpoint_dir(cfg, policy_id), f"{name_prefix}_*")
    checkpoint_dict = Learner.load_checkpoint(checkpoints, device)
    actor_critic.load_state_dict(checkpoint_dict["model"])
    return actor_critic


def get_expert_actions(obs, cfg: Config, actor_critic, env, env_info, device):
    rnn_states = torch.zeros([env.num_agents, get_rnn_size(cfg)], dtype=torch.float32, device=device)

    obs = {"obs": obs}
    with torch.no_grad():
        normalized_obs = prepare_and_normalize_obs(actor_critic, obs)
        policy_outputs = actor_critic(normalized_obs, rnn_states)

        # sample actions from the distribution by default
        actions = policy_outputs["actions"]
    return actions

## Load expert model

In [ ]:
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sf_examples.mujoco.mujoco_params import add_mujoco_env_args, mujoco_override_defaults
from sf_examples.mujoco.train_mujoco import register_mujoco_components
from sf_examples.mujoco.mujoco_utils import MUJOCO_ENVS, make_mujoco_env


def register_mujoco_components():
    for env in MUJOCO_ENVS:
        register_env(env.name, make_mujoco_env)


register_mujoco_components()
argv = ["--algo=APPO", "--env=mujoco_ant", "--experiment=sf_Ant", "--train_dir=train_dir", "--no_render"]
parser, partial_cfg = parse_sf_args(argv=argv, evaluation=True)
add_mujoco_env_args(partial_cfg.env, parser)
mujoco_override_defaults(partial_cfg.env, parser)
cfg = parse_full_cfg(parser, argv=argv)
expert = create_expert(cfg)

## Helpers (collecting data & evaluation)

In [ ]:
import time

from IPython import display as ipydisplay

import torch
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from matplotlib import animation


@torch.no_grad()
def run_policy (env, model, total_steps=10000, verbose=True):
    obs_array = np.empty([total_steps, *env.observation_space.shape])
    act_array = np.empty([total_steps, env.action_space.shape[0]])
    rew_array = np.empty([total_steps, 1])
    done_array = np.empty([total_steps, 1])

    iter_time = time.time()
    done = True
    for i in range(total_steps):
        if verbose and (i + 1) % 1000 == 0:
            steps_per_second = 1000 / (time.time() - iter_time)
            print(f'Step {i + 1}/{total_steps}, Steps per second: {steps_per_second}')
            iter_time = time.time()

        if done:
            obs, info = env.reset()

        act = model(torch.from_numpy(obs).unsqueeze(0).float())[0].detach().cpu().numpy()
        obs_, rew, terminated, truncated, _ = env.step(act)
        done = terminated or truncated

        obs_array[i] = obs
        act_array[i] = act
        rew_array[i] = rew
        done_array[i] = float(done)

        obs = obs_

    return obs_array, act_array, rew_array, done_array

def calculate_returns(rew, done):
    rew_cumsum = np.cumsum(rew)[:, None]
    ret_cumsum = rew_cumsum * done
    ret_cumsum_trimed = ret_cumsum[np.nonzero(ret_cumsum)]
    ret_cumsum_trimed[1:] -= ret_cumsum_trimed[:-1]
    return ret_cumsum_trimed

def evaluate_agent(env, model, verbose=False):
    _, _, rew, done = run_policy(env, model, total_steps=50000, verbose=verbose)
    rets = calculate_returns(rew, done)

    print(f'Num. episodes: {len(rets)}')
    print(f'Avg. return: {np.mean(rets)}')
    print(f'Max. return: {np.max(rets)}')
    print(f'Min. return: {np.min(rets)}')

@torch.no_grad()
def collect_frames(eval_env, model, num_frames=2000):
    state, _ = eval_env.reset()
    state = torch.from_numpy(np.array(state)).float()
    frames = []

    for _ in range(num_frames):
        frames.append(eval_env.render())

        action = model(state.unsqueeze(0))[0]
        next_state, reward, terminal, truncate, info = eval_env.step(action.detach().cpu().numpy())

        if terminal or truncate:
            state, _ = eval_env.reset()
        state = next_state
        state = torch.from_numpy(np.array(state)).float()

    return frames

def display_frames_as_video(frames):
    """
    Displays a list of frames as a video.
    """
    plt.figure(figsize=(frames[0].shape[1] / 72.0, frames[0].shape[0] / 72.0), dpi=72)
    patch = plt.imshow(frames[0])
    plt.axis('off')

    def animate(i):
        patch.set_data(frames[i])

    anim = animation.FuncAnimation(plt.gcf(), animate, frames=len(frames), interval=50)
    ipydisplay.display(ipydisplay.HTML(anim.to_jshtml()))

## 1. Behavior Clonning

Algorithm

1. Collect the expert data.
2. Fit the model (classifier/regressor) to the expert data.

### Create model

In [ ]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, input_shape, output_size, hidden_sizes=(256, 256), hidden_activation=nn.Tanh(), output_activation=None, l2_weight=0.0001):
        super(MLP, self).__init__()
        self.layers = nn.Sequential()

        # Input layer
        self.layers.add_module("input", nn.Linear(input_shape, hidden_sizes[0]))
        self.layers.add_module("input_activation", hidden_activation)

        # Hidden layers
        layer_sizes = zip(hidden_sizes[:-1], hidden_sizes[1:])
        for i, (h1, h2) in enumerate(layer_sizes):
            self.layers.add_module(f"hidden_{i}", nn.Linear(h1, h2))
            self.layers.add_module(f"activation_{i}", hidden_activation)

        # Output layer
        self.layers.add_module("output", nn.Linear(hidden_sizes[-1], output_size))
        if output_activation is not None:
            self.layers.add_module("output_activation", output_activation)

        # Regularization
        self.l2_weight = l2_weight

    def forward(self, x):
        # Forward pass through the network
        x = self.layers(x)
        return x

    def l2_regularization(self):
        l2_reg = None
        for name, param in self.named_parameters():
            if 'weight' in name:
                if l2_reg is None:
                    l2_reg = param.norm(2)
                else:
                    l2_reg = l2_reg + param.norm(2)
        return self.l2_weight * l2_reg

### Function for training the model

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def train(obs, act, model, num_epochs=10, batch_size=32):
    obs_tensor = torch.tensor(obs, dtype=torch.float32)
    act_tensor = torch.tensor(act, dtype=torch.float32)

    dataset = TensorDataset(obs_tensor, act_tensor)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Define the loss function and optimizer
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())

    # Training loop
    for epoch in range(num_epochs):
        for batch_idx, (x_batch, y_batch) in enumerate(data_loader):
            # Forward pass
            y_pred = model(x_batch)

            # Compute loss
            loss = loss_fn(y_pred, y_batch) + model.l2_regularization()

            # Zero gradients, perform a backward pass, and update the weights.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Print loss every epoch
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")

In [ ]:
env = gym.make('Ant-v4')
env.num_agents = 1
env_info = extract_env_info(env, cfg)
device = torch.device("cpu" if cfg.device == "cpu" else "cuda")
collected_data = run_policy(env, functools.partial(get_expert_actions, cfg=cfg, actor_critic=expert, env=env, env_info=env_info, device=device), total_steps=10000)

Step 1000/10000, Steps per second: 203.27740459643104
Step 2000/10000, Steps per second: 508.65043258832884
Step 3000/10000, Steps per second: 348.01601686950505
Step 4000/10000, Steps per second: 385.80725881366334
Step 5000/10000, Steps per second: 432.4563306440637
Step 6000/10000, Steps per second: 474.8108139438002
Step 7000/10000, Steps per second: 512.5131387651553
Step 8000/10000, Steps per second: 510.06068601230163
Step 9000/10000, Steps per second: 510.71989604181806
Step 10000/10000, Steps per second: 417.8592492229218


In [ ]:
obs, act, rewards, dones = collected_data

# EXERCISE: Create model
model = ...

train(...)

Epoch 1/10, Loss: 0.014443062245845795


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch 2/10, Loss: 0.017369214445352554
Epoch 3/10, Loss: 0.014867021702229977
Epoch 4/10, Loss: 0.0086848558858037
Epoch 5/10, Loss: 0.009866082109510899
Epoch 6/10, Loss: 0.01155081670731306
Epoch 7/10, Loss: 0.011811533942818642
Epoch 8/10, Loss: 0.01292515080422163
Epoch 9/10, Loss: 0.008926967158913612
Epoch 10/10, Loss: 0.009364139288663864


In [ ]:
evaluate_agent(env, model)

Num. episodes: 64
Avg. return: 2820.6208538041883
Max. return: 5599.613278955381
Min. return: 225.72703599976376


### Exercise

Discuss the questions

1. In principle, do we need the expert policy for BC?


1. What are the problems with BC?


1. How can we help BC do better?


In [ ]:
# Collect the exploratory data
def exploratory(obs, **kwargs):
    """Adds the Gaussian noise to the expert actions."""
    # TODO: Implement me
    action = ...
    return action


expl_data = run_policy(env, functools.partial(exploratory, cfg=cfg, actor_critic=expert, env=env, env_info=env_info, device=device), total_steps=10000)


Step 1000/10000, Steps per second: 334.8148789122067
Step 2000/10000, Steps per second: 400.7286733259493
Step 3000/10000, Steps per second: 398.78733321125276
Step 4000/10000, Steps per second: 383.1056969917513
Step 5000/10000, Steps per second: 339.3384770858358
Step 6000/10000, Steps per second: 380.83564029094316
Step 7000/10000, Steps per second: 400.0503221869435
Step 8000/10000, Steps per second: 401.80109566685
Step 9000/10000, Steps per second: 403.971579182274
Step 10000/10000, Steps per second: 251.64020746273323


In [ ]:
obs_expl, act_expl, rewards, dones = expl_data
# Exercise: Run BC on the exploratory data

# ANSWER
...
# END ANSWER

Epoch 1/10, Loss: 0.01816621422767639
Epoch 2/10, Loss: 0.01638553850352764
Epoch 3/10, Loss: 0.011592207476496696
Epoch 4/10, Loss: 0.011816447600722313
Epoch 5/10, Loss: 0.011506960727274418
Epoch 6/10, Loss: 0.011857861652970314
Epoch 7/10, Loss: 0.011195662431418896
Epoch 8/10, Loss: 0.011018062941730022
Epoch 9/10, Loss: 0.009818049147725105
Epoch 10/10, Loss: 0.011045064777135849


In [ ]:
evaluate_agent(env, model_expl)

Num. episodes: 61
Avg. return: 2889.012770137162
Max. return: 5418.8661584957445
Min. return: 15.435665271063044


### Exercise

Answer the questions

1. Why does it do better?
1. How can we use the expert to further improve the data?

In [ ]:
# Exercise: Infere the expert actions on the exploratory observations
#           and run BC on it.

# ANSWER
...
# ANSWER END

Epoch 1/10, Loss: 0.012582506984472275
Epoch 2/10, Loss: 0.010205846279859543
Epoch 3/10, Loss: 0.011247273534536362
Epoch 4/10, Loss: 0.013896957971155643
Epoch 5/10, Loss: 0.012700987979769707
Epoch 6/10, Loss: 0.010154202580451965
Epoch 7/10, Loss: 0.01089314091950655
Epoch 8/10, Loss: 0.008588881231844425
Epoch 9/10, Loss: 0.009306480176746845
Epoch 10/10, Loss: 0.013685030862689018


In [ ]:
evaluate_agent(env, model_expl2)

Num. episodes: 63
Avg. return: 3960.000221469911
Max. return: 5798.379262125047
Min. return: 217.6652887265882


### Exercise

Answer the questions

1. Did it help? Why?

1. How can you extend this idea?


## 2. Imitation Learning via Interactive Demostrator

[DAgger](https://www.ri.cmu.edu/pub_files/2011/4/Ross-AISTATS11-NoRegret.pdf)

1. Collect the expert data.
2. Fit the model (classifier/regressor) to the expert data.
3. Collect the imitator data.
4. Infere the expert actions on the imitator data.
5. Fit the model to the extended dataset.
6. Repeat from 3.

In [ ]:
# We will pre-train on less expert data to keep the same dataset size
obs_ = obs[:2000,:]
act_ = act[:2000,:]

# EXERCISE: pretrain on first 2000 samples
# ANSWER
...
# END ANSWER

evaluate_agent(env, model_dagger)

Epoch 1/10, Loss: 0.024661429226398468
Epoch 2/10, Loss: 0.015465846285223961
Epoch 3/10, Loss: 0.01956024207174778
Epoch 4/10, Loss: 0.012147408910095692
Epoch 5/10, Loss: 0.01844610832631588
Epoch 6/10, Loss: 0.015799645334482193
Epoch 7/10, Loss: 0.011363713070750237
Epoch 8/10, Loss: 0.01228661835193634
Epoch 9/10, Loss: 0.015529208816587925
Epoch 10/10, Loss: 0.012060150504112244
Num. episodes: 77
Avg. return: 2047.9112360615823
Max. return: 4609.092582294517
Min. return: 41.75504788161197


In [ ]:
# Exercise: Implement DAgger

for i in range(4):
    print(f'\n### Iter. {i+1} ###')

    # ANSWER


    # END ANSWER



### Iter. 1 ###

1. Data collection
Step 1000/2000, Steps per second: 1479.354901286494
Step 2000/2000, Steps per second: 1547.5857346108662

2. Training
Epoch 1/10, Loss: 0.016831206157803535
Epoch 2/10, Loss: 0.01534556970000267
Epoch 3/10, Loss: 0.013487616553902626
Epoch 4/10, Loss: 0.01240621693432331
Epoch 5/10, Loss: 0.011124205775558949
Epoch 6/10, Loss: 0.015226945281028748
Epoch 7/10, Loss: 0.014033282175660133
Epoch 8/10, Loss: 0.012436695396900177
Epoch 9/10, Loss: 0.013683512806892395
Epoch 10/10, Loss: 0.012340785004198551

3. Evaluation
Num. episodes: 55
Avg. return: 4268.532364189748
Max. return: 5185.424739415641
Min. return: 227.93642113429087

### Iter. 2 ###

1. Data collection
Step 1000/2000, Steps per second: 1701.6136164660768
Step 2000/2000, Steps per second: 1629.3178260244651

2. Training
Epoch 1/10, Loss: 0.014591274783015251
Epoch 2/10, Loss: 0.010312606580555439
Epoch 3/10, Loss: 0.010615671053528786
Epoch 4/10, Loss: 0.01181822456419468
Epoch 5/10, Loss: 

### Note

Training the expert with the PPO algorithm took 10M data samples (env. interactions). Here, we nearly match it with only 10k samples! Training from the expert can be much more efficient than reinforcement learning.